# TissueAgent cell-annotation benchmark

This demo prepares a leakage-free benchmark, runs TissueAgent, runs the selected direct baselines, and evaluates every query cell. It never installs packages or resets/deletes shared data directories. Mouse CNS is rebuilt from the required Zenodo raw-expression, spatial, metadata, and cluster CSVs; the existing derived mouse H5AD is not used.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'demo':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

DATASET = 'developing_human_heart'  # developing_human_heart | mouse_cns | mouse_spinal_cord | ovarian_cancer | human_skin_atlas
RUN_MODE = 'quick'                  # quick | full
METHODS = ['tissueagent', 'celltypist', 'gptcelltype']
print({'dataset': DATASET, 'run_mode': RUN_MODE, 'methods': METHODS})

## Prepare and validate

Preparation reuses checksum-verified local files. It downloads only a missing reference declared by the selected manifest. Large source data and ground truth are never sent to an LLM.

In [ ]:
from demo.cell_annotation.benchmarks import prepare_benchmark

prepared = prepare_benchmark(DATASET, run_mode=RUN_MODE)
prepared

## Run TissueAgent

The API key must already be inherited by this kernel. The notebook never displays or saves it.

In [ ]:
if 'tissueagent' in METHODS:
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY is not visible; restart Jupyter from the shell where it is exported.')
    from demo.cell_annotation.tissueagent_runner import run_tissueagent
    tissueagent_result = run_tissueagent(prepared)
    display(tissueagent_result)

## Run baselines directly

CellTypist and GPTCellType are method-specific evaluation code, not TissueAgent tools. The locked benchmark dependencies must be prepared before launching the notebook.

In [ ]:
from demo.cell_annotation.baselines import run_celltypist, run_gptcelltype

baseline_results = {}
if 'celltypist' in METHODS:
    baseline_results['celltypist'] = run_celltypist(prepared)
if 'gptcelltype' in METHODS:
    baseline_results['gptcelltype'] = run_gptcelltype(prepared)
baseline_results

## Evaluate

Unmapped, failed, or missing predictions are scored as `Unassigned`; no cells are dropped. Metrics, confusion matrices, normalized predictions, and unmapped-label audits are written to the run directory.

In [ ]:
from demo.cell_annotation.evaluation import evaluate_predictions

metrics = evaluate_predictions(prepared)
metrics